In [9]:
import sys
sys.path.append('..')
import torch
from torchvision import transforms
from PIL import Image
import glob

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

image_files = glob.glob("../data/HAM10000_images_part_1/*.jpg")
test_tensor = transform(
    Image.open(image_files[5]).convert("RGB")
).unsqueeze(0)

print("image loaded:", image_files[5])

image loaded: ../data/HAM10000_images_part_1\ISIC_0024311.jpg


In [10]:
from agents.shap_node import run_shap

# build a minimal state just for the shap node
test_state = {
    "image_tensor":   test_tensor,
    "pred_class_idx": 0,
    "confidence":     0.45
}

print("running SHAP node directly (not through full graph)...")
print("this tests the node in isolation before trusting the full graph\n")

shap_output = run_shap(test_state)
shap_result = shap_output['shap_result']

print("\nresult keys:", list(shap_result.keys()))
print("status:         ", shap_result['status'])
print("mean_shap:      ", shap_result['mean_shap'])
print("max_shap:       ", shap_result['max_shap'])
print("interpretation: ", shap_result['interpretation'])
print("plot generated: ", len(shap_result['plot_b64']) > 0)

running SHAP node directly (not through full graph)...
this tests the node in isolation before trusting the full graph

  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=success, mean=0.000761

result keys: ['status', 'mean_shap', 'max_shap', 'top_pixel_values', 'interpretation', 'shap_map', 'plot_b64']
status:          success
mean_shap:       0.000761
max_shap:        0.009579
interpretation:  Low feature attribution — the model's decision was distributed broadly across the image with no strong focal point
plot generated:  True


In [11]:
from agents.gradcam_node import run_gradcam

gradcam_output = run_gradcam(test_state)
gradcam_result = gradcam_output['gradcam_result']

print("status:           ", gradcam_result['status'])
print("attention region: ", gradcam_result['attention_region'])
print("max attention:    ", gradcam_result['max_attention'])
print("interpretation:   ", gradcam_result['interpretation'])
print("plot generated:   ", len(gradcam_result['plot_b64']) > 0)

  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
status:            success
attention region:  center of the lesion
max attention:     0.9936
interpretation:    The model's attention was broadly distributed across much of the image with high gradient signal strength. High-attention coverage: 30.9% of image area.
plot generated:    True


In [12]:
from agents.critic import run_critic, route_from_critic

critic_state = {
    "shap_result":   shap_result,
    "gradcam_result": gradcam_result,
    "confidence":    0.45,
    "loop_count":    0
}

critic_output = run_critic(critic_state)
print("critique:       ", critic_output['critique'])
print("contradictions: ", critic_output['contradictions'])
print("loop_count now: ", critic_output['loop_count'])

# test the router
routing_state = {**critic_state, **critic_output}
route = route_from_critic(routing_state)
print("router decision:", route)

  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=0, loop_count=0
critique:        SHAP and Grad-CAM results look consistent
contradictions:  []
loop_count now:  1
  [Critic router] no contradiction or max loops, going to narrator
router decision: narrator


In [13]:
from agents.narrator import run_narrator

narrator_state = {
    "prediction":    "nv",
    "confidence":    0.45,
    "shap_result":   shap_result,
    "gradcam_result": gradcam_result,
    "critique":      critic_output['critique'],
    "contradictions": critic_output['contradictions']
}

narrator_output = run_narrator(narrator_state)
print("explanation:")
print(narrator_output['explanation'])
print("\nconfidence note:", narrator_output['confidence_note'])

  [Narrator node] writing explanation...
  [Narrator node] explanation written (457 chars)
explanation:
The model predicted 'nv' with 45.0% confidence. SHAP analysis shows: Low feature attribution — the model's decision was distributed broadly across the image with no strong focal point. Grad-CAM shows the model focused on the center of the lesion. The model's attention was broadly distributed across much of the image with high gradient signal strength. High-attention coverage: 30.9% of image area. Critic review: SHAP and Grad-CAM results look consistent.

confidence note: High confidence - both tools agree, prediction is reliable


In [14]:
import importlib
import agents.graph as graph_module
importlib.reload(graph_module)
from agents.graph import xai_agent

full_state = {
    "image_path":      image_files[5],
    "image_tensor":    test_tensor,
    "prediction":      "nv",
    "confidence":      0.45,
    "pred_class_idx":  0,
    "shap_result":     {},
    "gradcam_result":  {},
    "critique":        "",
    "contradictions":  [],
    "next_action":     "",
    "explanation":     "",
    "confidence_note": "",
    "loop_count":      0
}

print("running full agent graph end to end...")
print("confidence=0.45 so SHAP will run first\n")

final = xai_agent.invoke(full_state)

print("\nfinal explanation:")
print(final['explanation'])
print("\nfinal confidence note:")
print(final['confidence_note'])
print(final['shap_result'])

XAI agent graph compiled successfully
running full agent graph end to end...
confidence=0.45 so SHAP will run first

  [Planner] confidence=0.45, loop=0
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=1, loop_count=0
    - SHAP analysis failed or returned error
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.45, loop=1
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=1, loop_cou

In [15]:
import base64
import matplotlib.pyplot as plt
from PIL import Image
import io

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

shap_img = Image.open(
    io.BytesIO(base64.b64decode(final['shap_result']['plot_b64']))
)
gradcam_img = Image.open(
    io.BytesIO(base64.b64decode(final['gradcam_result']['plot_b64']))
)

axes[0].imshow(shap_img)
axes[0].set_title("SHAP explanation")
axes[0].axis("off")

axes[1].imshow(gradcam_img)
axes[1].set_title("Grad-CAM explanation")
axes[1].axis("off")

plt.suptitle("XAI Agent — both tools ran and produced explanations", fontsize=13)
plt.tight_layout()
plt.show()

print("both plots visible means the full agent pipeline is working end to end")

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x000002476F63F290>

In [16]:
import mlflow

mlflow.set_experiment("xai-agent-runs")

with mlflow.start_run(run_name="day10-full-agent-test"):
    mlflow.log_param("confidence",    final['confidence'])
    mlflow.log_param("prediction",    final['prediction'])
    mlflow.log_param("next_action",   final['next_action'])
    mlflow.log_param("loop_count",    final['loop_count'])
    mlflow.log_metric("shap_mean",    final['shap_result'].get('mean_shap', 0))
    mlflow.log_metric("shap_max",     final['shap_result'].get('max_shap', 0))
    mlflow.log_metric("gradcam_max",  final['gradcam_result'].get('max_attention', 0))

    print("agent run logged to MLflow")
    print("open mlflow ui to see it: run 'mlflow ui' in a new terminal")

2026/07/28 22:14:07 INFO mlflow.tracking.fluent: Experiment with name 'xai-agent-runs' does not exist. Creating a new experiment.


agent run logged to MLflow
open mlflow ui to see it: run 'mlflow ui' in a new terminal
